In [ ]:
import spacy

import pandas as pd
import numpy as np

import textdescriptives as td

from dataset_evaluation.utils import add_column
from dataset_evaluation.evaluation_framework import EvaluationFramework

from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import ast

from vendi_score import text_utils

from diversity import (
	compression_ratio
)

from scipy.stats import (shapiro, kruskal, f_oneway)
import scikit_posthocs as sp
from cliffs_delta import cliffs_delta

import seaborn as sns
from matplotlib.patches import Patch

plt.style.use("seaborn-v0_8-whitegrid")

from dataset_evaluation.modelasametric.simlex999_comparison import (
    pairwise_spearman,
    select_invocab_simlex,
    compute_similarity,
    train_and_save_w2v
)
from scipy.stats import spearmanr

## Load datasets

### Baseline

In [ ]:
baseline_data_full = pd.read_csv('baseline_research_prompting_llama3-8b.csv')
baseline_data_full.rename(columns={"completion": "story"}, inplace=True)
baseline_data = baseline_data_full.copy()
baseline_data = baseline_data.sample(50)

### Fewshot

In [ ]:
fewshot_data = pd.read_csv('fewshot_600.csv')
fewshot_data.drop(columns=['Unnamed: 0'], inplace=True)
fewshot_data.rename(columns={'stories': "story"}, inplace=True)
fewshot_data

In [ ]:
path_name = 'fewshot.csv'
df_fewshot = pd.read_csv(path_name)
df_fewshot.rename(columns={'stories': 'story'}, inplace=True)
df_fewshot.columns

#### Persona

In [ ]:
theme_data = pd.read_csv('storytheme_research_prompting_llama3-8b.csv')

# columns containing completions
completion_cols = ["completion1", "completion2", "completion3", "completion4", "completion5"]

# ---- case 1: theme == 0 ----
df_persona = (
	theme_data[theme_data["theme"] == str(0)][completion_cols]
	.melt(value_name="story", var_name="completion_number")
	.drop(columns="completion_number")
	.reset_index(drop=True)
)

persona_data = pd.read_csv('persona_prompt_600.csv')
persona_data.drop(columns=['Unnamed: 0'], inplace=True)
persona_data.rename(columns={'stories': "story"}, inplace=True)
persona_data


#### Themes

In [ ]:
theme_data = pd.read_csv('storytheme_research_prompting_llama3-8b.csv')

# columns containing completions
completion_cols = ["completion1", "completion2", "completion3", "completion4", "completion5"]

# ---- case 2: theme != 0 ----
df_theme = (
	theme_data[theme_data["theme"] != str(0)][["model", "theme", "system", "user"] + completion_cols]
	.melt(id_vars=["model", "theme", "system", "user"],
		  value_name="story",
		  var_name="completion_number")
	.drop(columns="completion_number")
	.reset_index(drop=True)
)

themes_data_600 = pd.read_csv('themes_condition_600.csv')
themes_data_600.drop(columns=['Unnamed: 0'], inplace=True)
themes_data_600.rename(columns={'stories': "story"}, inplace=True)
themes_data_600


### Narrative features

In [ ]:
narrative_elements_data = pd.read_csv('storyelement_research_prompting_llama3-8b.csv')

# columns containing completions
completion_cols = ["completion1", "completion2", "completion3", "completion4", "completion5"]

# ---- case 2: narrative_element != 0 ----
df_elements = (
	narrative_elements_data[narrative_elements_data["narrative_element"] != str(0)][["model", "narrative_element", "system", "user"] + completion_cols]
	.melt(id_vars=["model", "narrative_element", "system", "user"],
		  value_name="story",
		  var_name="completion_number")
	.drop(columns="completion_number")
	.reset_index(drop=True)
)

df_elements = df_elements.sample(50)

story_element_data_600 = pd.read_csv('story_elements_condition_600.csv')
story_element_data_600.drop(columns=['Unnamed: 0'], inplace=True)
story_element_data_600.rename(columns={'stories': "story"}, inplace=True)
story_element_data_600

### Additional words

In [ ]:
awords_data = pd.read_csv('additionalwords_research_prompting_llama3-8b.csv')

# columns containing completions
completion_cols = ["completion1", "completion2", "completion3", "completion4", "completion5"]

# ---- case 2: theme != 0 ----
df_awords = (
	awords_data[awords_data["theme"] != str(0)][["model", "theme", "system", "user"] + completion_cols]
	.melt(id_vars=["model", "theme", "system", "user"],
		  value_name="story",
		  var_name="completion_number")
	.drop(columns="completion_number")
	.reset_index(drop=True)
)

awords_data_600 = pd.read_csv('chosen_words_condition_600.csv')
awords_data_600.drop(columns=['Unnamed: 0'], inplace=True)
awords_data_600.rename(columns={'stories': "story"}, inplace=True)
awords_data_600


### POS-tags

In [ ]:
postag_data = pd.read_csv('fixedpos_research_prompting_llama3-8b.csv')

# columns containing completions
completion_cols = ["completion1", "completion2", "completion3", "completion4", "completion5"]

# ---- case 2: theme != 0 ----
df_pos = (
	postag_data[postag_data["theme"] != str(0)][["model", "theme", "system", "user"] + completion_cols]
	.melt(id_vars=["model", "theme", "system", "user"],
		  value_name="story",
		  var_name="completion_number")
	.drop(columns="completion_number")
	.reset_index(drop=True)
)

pos_data_600 = pd.read_csv('pos_tags_condition_600.csv')
pos_data_600.drop(columns=['Unnamed: 0'], inplace=True)
pos_data_600.rename(columns={'stories': "story"}, inplace=True)
pos_data_600

### ChiSCor

In [ ]:
path_name = 'ChiSCor_master_df.csv'
df_chiscor = pd.read_csv(path_name, index_col=0)
df_chiscor = df_chiscor.rename(columns={'story_raw': 'story'})

In [ ]:
all_datasets = {'baseline': baseline_data, 'persona': df_persona, 'theme': df_theme, 'narrative-element': df_elements, 'a_words': df_awords, 'pos': df_pos, 'few_shot': df_fewshot, 'chiscor': df_chiscor}

In [ ]:
all_600_datasets = {'baseline': baseline_data_full, 'persona': persona_data, 'theme': themes_data_600, 'narrative-element': story_element_data_600, 'a_words': awords_data_600, 'pos': pos_data_600, 'few_shot': fewshot_data, 'chiscor': df_chiscor}

### Reference corpora

In [ ]:
ref_standard_dep = Path('datasets/BasiScript/BS_dep_lexicon.csv')
ref_standard_uni = Path('datasets/BasiScript/BS_unigram_lexicon.csv')
ref_standard_bi = Path('datasets/BasiScript/BS_bigram_lexicon.csv')

In [ ]:
ref_spoken_b_csv = Path('datasets/CGN/CGN_pos_bigram.csv')
ref_spoken_u_csv  = Path('datasets/CGN/CGN_pos_unigram.csv')
ref_spoken_t_csv = Path('datasets/CGN/CGN_pos_trigram.csv')

## Story quality

Overview used metrics:
- Coherence
	- Local contextuality
- Grammaticality
	- Grammaticality
- Surprise
	- Creative perplexity
- Diversity
	- Lexical diversity: self-bleu
	- lexical diversity: moving mtld
- Complexity
	- Lexical complexity: unique words
	- Lexical complexity: Average word length
	- Syntactic complexity: avg. components
	- Syntactic complexity: dependency distance
	- Syntactic complexity: syntactic tree depth

In [ ]:
eval_f = EvaluationFramework(language='nl',
							  pos_unigram=ref_spoken_u_csv, 
							  pos_bigram=ref_spoken_b_csv,
							  pos_trigram=ref_spoken_t_csv,
							  ref_unigram=ref_standard_uni,
							  ref_bigram=ref_standard_bi,
							  ref_ling_constrained=ref_standard_dep,
							  embedding_model='jegormeister/bert-base-dutch-cased')

In [ ]:
# Surprise: creative perplexity
eval_f.add_pipe('creative_perplexity_dep')
# Local contextuality
eval_f.add_pipe("local_contextuality")
# Grammaticality
eval_f.add_pipe('grammaticality')
# Diversity (lexical) self-bleu
eval_f.add_pipe('self-bleu')
# Diversity (lexical) moving mtld
eval_f.add_pipe('lexical_diversity')
# Complexity (lexical) unique words
eval_f.add_pipe('unique-words')
# Complexity (lexical) average word length
eval_f.add_pipe('avg-word-length')
# Complexity (syntactic)  average components
eval_f.add_pipe('average_components')
# Complexity (syntactic) dependency distance
eval_f.add_pipe('dependency_distance')
# Complexity (syntactic) syntactic tree depth
eval_f.add_pipe('syntactic_depth')
# Words before root
eval_f.add_pipe('wbr_average')

In [ ]:
def load_or_run_eval(eval_f, dataset, column, path_name, *, run=False):
	if Path(path_name).exists() and not run:
		df = pd.read_csv(path_name)
		if 'lexical_diversity' in df.columns:
			df['lexical_diversity'] = df['lexical_diversity'].apply(ast.literal_eval)
			
		return df

	dataset = eval_f.run_pipeline_on_df(dataset, column)
	dataset.to_csv(path_name, index=False)
	return dataset

In [ ]:
temp = all_datasets
for name, data in temp.items():
	print(f"Current method: {name}")
	generated_eval_results = load_or_run_eval(eval_f, data, 'story', f'results/metric_results/eval_results_{name}.csv')
	all_datasets[name] = generated_eval_results

### Plotting story quality

In [ ]:
def get_value_from_dictionary(x, key):
	new = []

	for item in x:
		new.append(item[key])
	return new

In [ ]:
metrics = ['creative_perplexity_dep', 'local_contextuality', 'grammaticality', 'self-bleu', 'lexical_diversity', 'unique-words', 'avg-word-length', 'average_components', 'dependency_distance', 'syntactic_depth']
story_metrics_means = {}
story_metrics_std = {}

for name in all_datasets.keys():
	story_metrics_means[name] = defaultdict(float)
	story_metrics_std[name] = defaultdict(float)

for name, data in all_datasets.items():
	for column in data.columns:
		if column in metrics:
			try:
				story_metrics_means[name][column] = data[column].mean()
				story_metrics_std[name][column] = data[column].std()
			except:
				x = get_value_from_dictionary(data[column], 'moving_mtld')
				story_metrics_means[name][column] = np.mean(np.array(x))
				story_metrics_std[name][column] = np.std(np.array(x))
story_metrics_means
df_story_statistics = pd.DataFrame.from_dict(story_metrics_means, orient="index")

## Statistical tests

#### Statistical local contextuality

In [ ]:
for name, group in zip(
	["chiscor", "baseline", "persona", 'theme', 'narrative-element', 'a_words', 'pos', 'few_shot', 'chiscor'], 
	[all_datasets['chiscor']['local_contextuality'], all_datasets['baseline']['local_contextuality'], all_datasets['persona']['local_contextuality'], all_datasets['theme']['local_contextuality'],
	 all_datasets['narrative-element']['local_contextuality'], all_datasets['a_words']['local_contextuality'], all_datasets['pos']['local_contextuality'], all_datasets['few_shot']['local_contextuality']]
):
	stat, p = shapiro(group)
	print(f"Group {name} normality p-value: {p:.4f}")

In [ ]:
H, p = kruskal(all_datasets['chiscor']['local_contextuality'], all_datasets['baseline']['local_contextuality'], all_datasets['persona']['local_contextuality'], all_datasets['theme']['local_contextuality'],
	 all_datasets['narrative-element']['local_contextuality'], all_datasets['a_words']['local_contextuality'], all_datasets['pos']['local_contextuality'], all_datasets['few_shot']['local_contextuality'])
print(f"H-statistic: {H:.4f}, p-value: {p}")

In [ ]:
data = [all_datasets['chiscor']['local_contextuality'], all_datasets['baseline']['local_contextuality'], all_datasets['persona']['local_contextuality'], all_datasets['theme']['local_contextuality'],
	 all_datasets['narrative-element']['local_contextuality'], all_datasets['a_words']['local_contextuality'], all_datasets['pos']['local_contextuality'], all_datasets['few_shot']['local_contextuality']]

# Dunn's test with Bonferroni correction
p_values = sp.posthoc_dunn(
	data,
	p_adjust="bonferroni"
)

print(p_values)

In [ ]:
delta, size = cliffs_delta(all_datasets['chiscor']['local_contextuality'], all_datasets['few_shot']['local_contextuality'])
print(f"Cliff's delta = {delta:.3f}, magnitude = {size}")

#### Statistical grammaticality

In [ ]:
for name, group in zip(
	["chiscor", "baseline", "persona", 'theme', 'narrative-element', 'a_words', 'pos', 'few_shot', 'chiscor'], 
	[all_datasets['chiscor']['grammaticality'], all_datasets['baseline']['grammaticality'], all_datasets['persona']['grammaticality'], all_datasets['theme']['grammaticality'],
	 all_datasets['narrative-element']['grammaticality'], all_datasets['a_words']['grammaticality'], all_datasets['pos']['grammaticality'], all_datasets['few_shot']['grammaticality']]
):
	stat, p = shapiro(group)
	print(f"Group {name} normality p-value: {p:.4f}")

In [ ]:
H, p = kruskal(all_datasets['chiscor']['grammaticality'], all_datasets['baseline']['grammaticality'], all_datasets['persona']['grammaticality'], all_datasets['theme']['grammaticality'],
	 all_datasets['narrative-element']['grammaticality'], all_datasets['a_words']['grammaticality'], all_datasets['pos']['grammaticality'], all_datasets['few_shot']['grammaticality'])
print(f"H-statistic: {H:.4f}, p-value: {p}")

In [ ]:
F, p = f_oneway(all_datasets['chiscor']['grammaticality'], all_datasets['baseline']['grammaticality'], all_datasets['persona']['grammaticality'], all_datasets['theme']['grammaticality'],
	 all_datasets['narrative-element']['grammaticality'], all_datasets['a_words']['grammaticality'], all_datasets['pos']['grammaticality'], all_datasets['few_shot']['grammaticality'])

print(f"F-statistic: {F:.4f}")
print(f"p-value: {p:.4f}")

In [ ]:
data = [all_datasets['chiscor']['grammaticality'], all_datasets['baseline']['grammaticality'], all_datasets['persona']['grammaticality'], all_datasets['theme']['grammaticality'],
	 all_datasets['narrative-element']['grammaticality'], all_datasets['a_words']['grammaticality'], all_datasets['pos']['grammaticality'], all_datasets['few_shot']['grammaticality']]

# Dunn's test with Bonferroni correction
p_values = sp.posthoc_dunn(
	data,
	p_adjust="bonferroni"
)

print(p_values)

In [ ]:
delta, size = cliffs_delta(all_datasets['baseline']['grammaticality'], all_datasets['few_shot']['grammaticality'])
print(f"Cliff's delta = {delta:.3f}, magnitude = {size}")

## Dataset quality

### Compression and Vendi

In [ ]:
dataset_statistics = {}
for name in all_datasets.keys():
	dataset_statistics[name] = defaultdict(float)

for name, data in all_datasets.items():
	embed_vs = text_utils.embedding_vendi_score(list(map(str, data['story'].to_list())), model_path='jegormeister/bert-base-dutch-cased')
	dataset_statistics[name]['vendi'] = embed_vs
	comp_score = compression_ratio(data['story'].to_list(), algorithm='gzip')
	dataset_statistics[name]['compression'] = comp_score
df_dataset_statistics = pd.DataFrame.from_dict(dataset_statistics, orient="index")

## SimLex-999

In [ ]:
simlex999 = pd.read_csv('dataset_evaluation/modelasametric/SimLex-999-Dutch-final.txt', names=['lemma1','lemma2','SimLexScore','POS'], skiprows=1, sep='\t')
simlex999.drop(index=364, inplace=True)

In [ ]:
# initialise word2vec model
mincount = 10
trained_models = {}

for name, data in all_600_datasets.items():
	model = train_and_save_w2v(data['story'], f'dataset_evaluation/modelasametric/story_Word2Vec_{name}_mincount{mincount}.kv', mincount=mincount)
	trained_models[name] = model

In [ ]:
best_datasets_models = {name: model for name, model in trained_models.items() if name in ['chiscor', 'baseline', 'few_shot', 'a_words']}
best_datasets_models


In [ ]:
df_filtered_simlex, filtered_simlex_list, filtered_simlex_tuple = select_invocab_simlex(simlex999, best_datasets_models.values())
len(filtered_simlex_tuple)

In [ ]:
cosine_similarities = {}
for name, model in best_datasets_models.items():
	cosine_similarities[name] = compute_similarity(filtered_simlex_tuple, model)

cosine_similarities['simlex999'] = df_filtered_simlex['SimLexScore']
statistic_results = pairwise_spearman(cosine_similarities)
statistic_results

In [ ]:
comparison_simlex = {}
for key, value in statistic_results.items():
	if key[0] == 'simlex999' or key[1] == 'simlex999':
		comparison_simlex[key] = value
comparison_simlex

In [ ]:
cosine_similarities = {}
chosen_lemmas = {}
results = {}

for name, model in best_datasets_models.items():
	df_filtered_simlex, filtered_simlex_list, filtered_simlex_tuple = select_invocab_simlex(simlex999, [model], sample=45)
	chosen_lemmas[name] = df_filtered_simlex
	cosine_similarities[name] = compute_similarity(filtered_simlex_tuple, model)
	r, p = spearmanr(cosine_similarities[name], df_filtered_simlex['SimLexScore'])
	results[(name, 'simlex')] = {"rho": r, "p": p}

results

## Age groups

In [ ]:
def agegroup_strings(value):

	if value == 1:
		return '4 - 6'
	if value == 3:
		return '6 - 7'
	if value == 4:
		return '7 - 8'
	if value == 5:
		return '8 - 9'
	if value == 6:
		return '9 - 10'
	if value == 7:
		return '10 - 11'
	if value == 8:
		return '11 - 12'

add_column('agegroup', 'agegroup_str', agegroup_strings, df_chiscor)

df_chiscor.agegroup_str = pd.Categorical(
	df_chiscor.agegroup_str,
	categories=['4 - 6', '6 - 7', '7 - 8', '8 - 9', '9 - 10', '10 - 11', '11 - 12'],
	ordered=True
)

In [ ]:
def different_stripe(x):
	first = str(x).split("_")[0]
	second = str(x).split("_")[1]
	return f'{first} - {second}'

age_llama = pd.read_csv('age_research_DD_llama3-8b.csv')
age_llama.drop(columns=['Unnamed: 0'], inplace=True)
age_llama['age'] = age_llama['age'].apply(lambda age: different_stripe(age))
age_llama

In [ ]:
age_llama['DD'] = age_llama.apply(lambda row: eval_f.run_component('dependency_distance', row['story']), axis = 1) 

In [ ]:
# --- 1) Sample max 50 per age group for generated stories (age_llama) ---
df_llama = (
	age_llama.groupby("age", group_keys=False)
	  .apply(lambda x: x.sample(n=min(50, len(x)), random_state=42))
	  .loc[:, ["age", "DD"]]
	  .rename(columns={"age": "age_group", "DD": "dep_dist"})
)
df_llama["source"] = "Generated stories"

# --- 2) Keep COMPLETE ChiSCor data (no mean aggregation) ---
df_chi = (
	df_chiscor.loc[:, ["agegroup_str", "dep_dist"]]
	  .rename(columns={"agegroup_str": "age_group"})
)
df_chi["source"] = "ChiSCor"

# --- 3) Combine for plotting ---
df_plot = pd.concat([df_llama, df_chi], ignore_index=True)

# --- 4) Sort age groups numerically by lower bound (works for "3_4", "10_12", etc.) ---
def lower_bound(x):
	return int(str(x).split("-")[0])

age_order = sorted(df_plot["age_group"].dropna().unique(), key=lower_bound)
print(df_plot)
# --- 5) Plot: raw points (with small separation between datasets) + point plot means ---
plt.figure(figsize=(10, 6))

# dodge=True creates the small horizontal separation between the two sources
sns.stripplot(
	data=df_plot,
	x="age_group", y="dep_dist",
	hue="source",
	order=age_order,
	dodge=True,          # separation between datasets at each age group
	jitter=0.15,         # small jitter within each dataset
	alpha=0.35,
	size=3
)

# Mean trend lines as a Seaborn point plot (solid circles + solid line)
sns.pointplot(
	data=df_plot,
	x="age_group", y="dep_dist",
	hue="source",
	order=age_order,
	dodge=0.35,          # match separation visually
	estimator=np.mean,
	errorbar=("ci", 95),       # change to ("ci", 95) or ("se") if you want error bars
	markers="o",
	linestyles="-",
	capsize=0
)

# Reference line
plt.axhline(y=2.5, color="tab:green", linestyle="--", linewidth=2,
			label="y = Mean Dependency Dist. Dutch")

# Aesthetics
plt.xlabel("Age Group", fontsize=16, fontweight='bold')
plt.ylabel("Dependency Distance", fontsize=16, fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()

# De-duplicate legend entries (because we used hue twice)
handles, labels = plt.gca().get_legend_handles_labels()
seen = set()
new_handles, new_labels = [], []
for h, l in zip(handles, labels):
	if l not in seen:
		new_handles.append(h)
		new_labels.append(l)
		seen.add(l)

plt.legend(new_handles, new_labels, title=None, fontsize=14)

ax = plt.gca()

# Make tick labels bigger
ax.tick_params(axis='both', which='major', labelsize=14)

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight('bold')

plt.savefig('results/paper_plots/dd_age.pdf')
